In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

from pathlib import Path

dataset_root = Path("LP IV Updated Datasets")
if not dataset_root.is_dir():
    dataset_root = Path("../LP IV Updated Datasets")
print("Dataset folder:", dataset_root.resolve())


In [ ]:
# Load and preprocess data
df = pd.read_csv(dataset_root / "AutoEncoder/CreditCard/creditcard.csv")
print("Loaded data shape:", df.shape)
print("First three rows:")
print(df.head(3))
print("Label counts:")
print(df.Class.value_counts().sort_index())

train, test = train_test_split(df, test_size=0.2, stratify=df.Class)

scaler = StandardScaler()
x_train = scaler.fit_transform(train[train.Class == 0].drop(columns="Class"))
x_test = scaler.transform(test.drop(columns="Class"))

print("Training input shape:", x_train.shape)
print("Test input shape:", x_test.shape)
print("Training value range:", x_train.min(), "to", x_train.max())
print("Test value range:", x_test.min(), "to", x_test.max())
print("First training signal: first ten feature values")
print(x_train[0, :10])


In [ ]:
# Create autoencoder
model = Sequential([
    Input(shape=(30,)),
    Dense(8, activation='relu'),
    Dense(30)
])

# Compile
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)
model.summary()


In [ ]:
# Train using normal inputs as their own reconstruction targets
history = model.fit(x_train, x_train, epochs=10, batch_size=128)
print("Last three training epochs:")
print(pd.DataFrame(history.history).tail(3))


In [ ]:
# Reconstruction errors for normal training inputs
train_prediction = model.predict(x_train)
train_error = np.mean((x_train - train_prediction) ** 2, axis=1)
print("First five training errors:", train_error[:5])
print("Training error min / mean / max:", train_error.min(), train_error.mean(), train_error.max())
threshold = np.percentile(train_error, 99.5)
print("Anomaly threshold (99.5th percentile):", threshold)


In [ ]:
# Reconstruction errors for test inputs
test_prediction = model.predict(x_test)
test_error = np.mean((x_test - test_prediction) ** 2, axis=1)
predictions = test_error > threshold
print("First ten test errors:", test_error[:10])
print("Test error min / mean / max:", test_error.min(), test_error.mean(), test_error.max())
print("First ten detected flags:", predictions[:10])
print("Fraud Detected:", predictions.sum())
print("First ten actual and predicted fraud labels:")
print(pd.DataFrame({
    "Actual fraud": test.Class.to_numpy()[:10],
    "Detected fraud": predictions[:10],
    "Error": test_error[:10]
}))
print(classification_report(test.Class, predictions, zero_division=0))
